# OpenAI Agents SDK 核心用法（带看懂版）

> 目标：带你看懂 OpenAI Agents SDK 的核心机制。全书用客服路由场景贯穿：单 Agent 工具调用、handoff 接管、Agent as tool 委托、context 注入。
> 核心示例使用 Scripted Model，不需要 API Key；真实模型示例单独标记。
> 基于 `openai-agents 0.21.1` / Python 3.12（2026-08-18）验证。

## 框架总览：Agent + Runner + Tools / Handoffs

```mermaid
flowchart LR
    A[Runner.run（启动执行）] --> B[Agent（当前配置）]
    B --> C[Model（模型调用）]
    C -->|function call（工具调用）| D[Tool（执行函数）]
    D --> C
    C -->|handoff（转交）| E[Specialist Agent（专家接管）]
    E --> C
    C -->|final output（最终输出）| F[RunResult（运行结果）]
```

`Runner` 管循环；`Agent` 只描述当前 Agent 能做什么；模型决定调用工具、转交 Agent，还是直接结束。

## 核心概念速查

| 概念 | 作用 | 本 Notebook 的位置 |
| --- | --- | --- |
| `Agent` | 配置 instructions、tools、handoffs、model | 所有例子 |
| `Runner` | 执行模型 → 工具 / handoff → 模型循环 | 所有例子 |
| `function_tool` | 把 Python 函数变成工具 schema | 例子一 |
| `handoffs` | 让另一个 Agent 接管对话 | 例子二 |
| `as_tool()` | 把另一个 Agent 当工具调用，原 Agent 继续掌控 | 例子三 |
| `RunContextWrapper` | 将业务 context 注入工具 | 例子四 |
| `RunResult` | 读取 final_output、last_agent、new_items | 例子一至三 |

## 概念关系：客服路由场景

```mermaid
flowchart LR
    U[用户问题] --> R[Runner（执行器）]
    R --> T[TriageAgent（客服分流）]
    T --> W[get_weather（天气工具）]
    T --> H[SpanishAgent（西语客服）]
    T --> A[ResearchAgent（研究 Agent as tool）]
    W --> T
    H --> U
    A --> T
    T --> U
```

| SDK 概念 | 客服场景中的位置 |
| --- | --- |
| `Runner` | 接收用户问题并推进整个循环 |
| `Agent` | `TriageAgent`、`SpanishAgent`、`ResearchAgent` |
| `function_tool` | `get_weather`，执行确定性业务动作 |
| `handoff` | Triage 把对话所有权交给 SpanishAgent |
| `as_tool` | Triage 调 ResearchAgent 获取报告，但自己继续回复 |
| `context` | 当前用户 ID、会员等级等业务数据 |

## 先准备一个无 Key 的 Scripted Model

真实 SDK 会把模型返回的 Responses API output 交给 `Runner`。为了不依赖 Key，下面的测试模型按预先写好的顺序返回文本或 function call。
它不模拟模型推理，只让我们看清 Runner 后续如何执行。

In [1]:
import json
from dataclasses import dataclass

from agents import (
    Agent,
    Model,
    ModelResponse,
    RunContextWrapper,
    Runner,
    Usage,
    function_tool,
    set_tracing_disabled,
)
from openai.types.responses import (
    ResponseFunctionToolCall,
    ResponseOutputMessage,
    ResponseOutputText,
)

set_tracing_disabled(True)


class ScriptedModel(Model):
    """测试用模型：按顺序返回固定的 Responses API 输出。"""

    def __init__(self, responses):
        self.responses = list(responses)
        self.index = 0

    async def get_response(
        self,
        system_instructions,
        input,
        model_settings,
        tools,
        output_schema,
        handoffs,
        tracing,
        *,
        previous_response_id,
        conversation_id,
        prompt,
    ):
        if self.index >= len(self.responses):
            raise RuntimeError("ScriptedModel 没有预设更多响应")
        response = self.responses[self.index]
        self.index += 1
        return response

    def stream_response(self, *args, **kwargs):
        raise NotImplementedError("本 Notebook 只验证非流式 Runner 循环")


def text_response(text: str, response_id: str) -> ModelResponse:
    return ModelResponse(
        output=[
            ResponseOutputMessage(
                id=response_id,
                content=[
                    ResponseOutputText(
                        annotations=[],
                        text=text,
                        type="output_text",
                    )
                ],
                role="assistant",
                status="completed",
                type="message",
            )
        ],
        usage=Usage(),
        response_id=response_id,
    )


def tool_response(name: str, arguments: dict, response_id: str, call_id: str) -> ModelResponse:
    return ModelResponse(
        output=[
            ResponseFunctionToolCall(
                id=f"{call_id}_item",
                call_id=call_id,
                name=name,
                arguments=json.dumps(arguments, ensure_ascii=False),
                type="function_call",
                status="completed",
            )
        ],
        usage=Usage(),
        response_id=response_id,
    )

print("Scripted Model ready")

Scripted Model ready


## 例子一：单 Agent + function_tool

这是 SDK 最小闭环。`Agent` 声明工具，`Runner` 负责把模型的 function call 变成真正的 Python 函数调用。

```mermaid
flowchart LR
    A[用户提问] --> B[Runner.run（启动）]
    B --> C[客服 Agent（模型调用）]
    C -->|get_weather| D[天气工具（函数执行）]
    D --> C
    C --> E[RunResult.final_output（最终回答）]
```

In [2]:
@function_tool
def get_weather(city: str) -> str:
    """查询城市天气。"""
    return f"{city}: 晴朗，25 度"


weather_model = ScriptedModel([
    tool_response("get_weather", {"city": "北京"}, "weather-1", "weather-call"),
    text_response("北京天气晴朗，25 度。", "weather-2"),
])

weather_agent = Agent(
    name="WeatherAgent",
    instructions="使用天气工具回答用户问题。",
    tools=[get_weather],
    model=weather_model,
)

weather_result = await Runner.run(weather_agent, "北京天气怎么样？")
print("最终回答:", weather_result.final_output)
print("当前 Agent:", weather_result.last_agent.name)
print("运行轨迹:", [type(item).__name__ for item in weather_result.new_items])

最终回答: 北京天气晴朗，25 度。
当前 Agent: WeatherAgent
运行轨迹: ['ToolCallItem', 'ToolCallOutputItem', 'MessageOutputItem']


#### 例子一：按执行顺序拆解

| 顺序 | 发生什么 | SDK 中的对象 |
| --- | --- | --- |
| 1 | `Runner.run(weather_agent, 用户问题)` | Runner 创建一次运行 |
| 2 | Scripted Model 返回 `get_weather` function call | `ModelResponse` |
| 3 | SDK 校验参数并执行 Python 函数 | `FunctionTool` |
| 4 | 工具输出被追加为 function call output | `ToolCallOutputItem` |
| 5 | SDK 再调用模型 | 同一个 `WeatherAgent` |
| 6 | 模型返回文本，循环结束 | `RunResult.final_output` |

关键认知：**`Agent` 不等于一次模型调用**。只要模型返回工具调用，`Runner` 就会继续推进；`Agent` 是“配置”，`Runner` 才是“执行循环”。

## 例子二：handoff（把对话所有权转交给专家）

客服分流是 handoff 的典型场景：TriageAgent 判断用户需要西班牙语服务后，把整个对话交给 SpanishAgent。handoff 完成后，目标 Agent 变成当前 Agent。

```mermaid
flowchart LR
    A[用户请求西班牙语服务] --> B[TriageAgent（分流）]
    B -->|transfer_to_spanishagent| C[SpanishAgent（接管）]
    C --> D[西语最终回答]
```

In [3]:
spanish_model = ScriptedModel([
    text_response("Hola, ¿en qué puedo ayudarte?", "spanish-1"),
])

spanish_agent = Agent(
    name="SpanishAgent",
    handoff_description="处理西班牙语客服问题。",
    instructions="始终使用西班牙语回答。",
    model=spanish_model,
)

# Agent 直接放进 handoffs 后，SDK 自动生成 transfer_to_spanishagent 工具。
triage_model = ScriptedModel([
    tool_response("transfer_to_spanishagent", {}, "triage-1", "handoff-call"),
])

triage_agent = Agent(
    name="TriageAgent",
    instructions="判断是否需要转交西语客服。",
    handoffs=[spanish_agent],
    model=triage_model,
)

handoff_result = await Runner.run(triage_agent, "请用西班牙语问候我")
print("最终回答:", handoff_result.final_output)
print("接管后的 Agent:", handoff_result.last_agent.name)
print("运行轨迹:", [type(item).__name__ for item in handoff_result.new_items])

最终回答: Hola, ¿en qué puedo ayudarte?
接管后的 Agent: SpanishAgent
运行轨迹: ['HandoffCallItem', 'HandoffOutputItem', 'MessageOutputItem']


#### 例子二：按执行顺序拆解

| 顺序 | 发生什么 | 关键点 |
| --- | --- | --- |
| 1 | Runner 从 `TriageAgent` 开始 | 当前 Agent 是 TriageAgent |
| 2 | 模型调用 `transfer_to_spanishagent` | handoff 对模型表现为一个特殊工具 |
| 3 | SDK 执行 handoff | 目标 Agent 变成 SpanishAgent |
| 4 | SpanishAgent 收到对话上下文并回答 | 新 Agent 继续 Runner 循环 |
| 5 | 返回最终结果 | `last_agent` 是 SpanishAgent |

handoff 的本质是：**转移对话控制权**。这和“调用一个工具拿结果”不一样。

## 例子三：Agent as tool（调用专家，但不交出控制权）

如果 Manager 需要先找 ResearchAgent 调研，然后自己统一汇总，就不应该用 handoff，而应该用 `research_agent.as_tool()`。

```mermaid
flowchart LR
    A[用户问题] --> B[Manager（主 Agent）]
    B -->|research tool| C[ResearchAgent（嵌套执行）]
    C -->|报告结果| B
    B --> D[Manager 汇总回答]
```

In [4]:
researcher_model = ScriptedModel([
    text_response("研究结果：北京今天晴朗。", "research-1"),
])
research_agent = Agent(
    name="ResearchAgent",
    instructions="研究问题并返回简洁报告。",
    model=researcher_model,
)

research_tool = research_agent.as_tool(
    tool_name="research",
    tool_description="研究用户问题并返回报告。",
)

manager_model = ScriptedModel([
    tool_response("research", {"input": "北京天气"}, "manager-1", "research-call"),
    text_response("汇总：北京今天晴朗。", "manager-2"),
])
manager_agent = Agent(
    name="ManagerAgent",
    instructions="必要时调用研究工具，再向用户汇总。",
    tools=[research_tool],
    model=manager_model,
)

manager_result = await Runner.run(manager_agent, "请调研北京天气")
print("最终回答:", manager_result.final_output)
print("仍由谁控制:", manager_result.last_agent.name)
print("运行轨迹:", [type(item).__name__ for item in manager_result.new_items])

最终回答: 汇总：北京今天晴朗。
仍由谁控制: ManagerAgent
运行轨迹: ['ToolCallItem', 'ToolCallOutputItem', 'MessageOutputItem']


#### 例子三：handoff 和 Agent as tool 对比

| 对比项 | handoff | `agent.as_tool()` |
| --- | --- | --- |
| 调用形式 | 特殊 handoff 工具 | 普通 function tool |
| 上下文 | 目标 Agent 默认接收对话历史 | 目标 Agent 接收构造出来的工具输入 |
| 控制权 | 目标 Agent 接管 | 原 Agent 保留 |
| `last_agent` | 变成目标 Agent | 仍是原 Agent |
| 典型场景 | 客服分流、专家接管 | 调研、计算、子任务后统一汇总 |

这是 OpenAI Agents SDK 多 Agent 设计里最值得记住的区别。

## 例子四：context 注入工具

业务数据不应该全部拼进用户文本。SDK 可以把一个普通 Python 对象作为 `context` 传给 Runner，再通过 `RunContextWrapper` 注入工具。

```mermaid
flowchart LR
    A[Runner.run(context=UserContext)] --> B[Agent]
    B --> C[get_member_level(ctx)]
    C --> D[读取 ctx.context.user_id]
    D --> B
```

In [5]:
@dataclass
class UserContext:
    user_id: str
    member_level: str


@function_tool
def get_member_level(ctx: RunContextWrapper[UserContext]) -> str:
    """查询当前用户的会员等级。"""
    return f"用户 {ctx.context.user_id} 是 {ctx.context.member_level} 会员"


context_model = ScriptedModel([
    tool_response("get_member_level", {}, "context-1", "context-call"),
    text_response("你是 user-42 的 Gold 会员。", "context-2"),
])
context_agent = Agent(
    name="ContextAgent",
    instructions="使用会员工具回答问题。",
    tools=[get_member_level],
    model=context_model,
)

context_result = await Runner.run(
    context_agent,
    "我的会员等级是什么？",
    context=UserContext(user_id="user-42", member_level="Gold"),
)
print("最终回答:", context_result.final_output)

最终回答: 你是 user-42 的 Gold 会员。


#### 例子四：按执行顺序拆解

| 顺序 | 发生什么 |
| --- | --- |
| 1 | `Runner.run(..., context=UserContext(...))` 保存本次运行上下文 |
| 2 | 模型请求 `get_member_level` | 工具参数 schema 中不需要暴露 user_id |
| 3 | SDK 把 `RunContextWrapper` 传给工具 | 工具从 `ctx.context` 读取业务对象 |
| 4 | 工具返回会员等级 | 结果回到模型 |

`context` 是调用侧传入的运行时依赖，不等同于用户消息，也不需要让模型自己生成。

## 真实模型调用（需要 API Key）

前面的 Scripted Model 只用于看懂 SDK 的执行机制。接真实模型时，通常不需要自己实现 `Model`：SDK 会使用默认 OpenAI 模型。

In [ ]:
from agents import Agent, Runner

agent = Agent(
    name="Assistant",
    instructions="你是一个简洁的客服助手。",
)
result = await Runner.run(agent, "请用一句话介绍你自己。")
print(result.final_output)

## 能力总览

| 能力 | 关键 API | 本 Notebook |
| --- | --- | --- |
| Agent 循环 | `Runner.run` / `run_sync` | 例子一 |
| 函数工具 | `@function_tool` | 例子一、四 |
| 多 Agent 接管 | `handoffs` / `handoff()` | 例子二 |
| Agent 嵌套调用 | `agent.as_tool()` | 例子三 |
| 运行时业务上下文 | `context` / `RunContextWrapper` | 例子四 |
| 会话记忆 | `SQLiteSession` / `Session` | 进阶 |
| 结构化输出 | `output_type` | 进阶 |
| 防护 | `input_guardrails` / `output_guardrails` | 进阶 |
| 外部工具 | MCP / Hosted Tools | 进阶 |
| 可观测性 | Tracing | 进阶 |

## 小结：什么时候选 OpenAI Agents SDK

- 想快速得到一个**单 Agent + 工具**闭环：选它。
- 想做客服分流、专家接管：重点理解 `handoff`。
- 想让主 Agent 调多个子 Agent 后自己汇总：重点理解 `agent.as_tool()`。
- 需要显式状态图、复杂并行、断点恢复和精确路由：回到 LangGraph。

核心记忆点：**Agent 是配置，Runner 是循环；handoff 是接管，as_tool 是委托。**